In [0]:
# FIXME
dbutils.widgets.text("taskValue", "dummy")
try:
    taskValue =  dbutils.widgets.get("taskValue")
    print ("task value  from dyn parameters ", taskValue)  #  {{job.name}}-{{job.run_id}}
except:
    print ("error in getting task value")
    pass

In [0]:

movies_path = dbutils.jobs.taskValues.get(taskKey = "movies-bronze-to-silver", key = "movies_silver_path", debugValue = "movies-not-found")

In [0]:
# widgets are used for parameters  at jobs clusters/workflow
dbutils.widgets.text("myname", "Gopal") # combobox, multiselect, select
name = dbutils.widgets.get("myname")  # read data from widget, getAll, returns dict key/value
print ("name", name)

name Gopal


In [0]:
# remove widget by name
# dbutils.widgets.remove("myname")
# rmeove all widgets
dbutils.widgets.removeAll()
     

In [0]:
# Convert CSV files into PArquet format
# Delta Lake: Move data from Bronze to Silver
# Row based CSV foramt to Column based Parquet

# download small -  1 MB, medium - 350 mb

# download small dataset https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
 
# large one only if you want to explore else no https://files.grouplens.org/datasets/movielens/ml-latest.zip
 
dbutils.widgets.text("moviesPath", "abfss://bronze@gopaldatalake2.dfs.core.windows.net/movielens/movies")
# target must be silver
dbutils.widgets.text("moviesTargetPath", "abfss://silver@gopaldatalake2.dfs.core.windows.net/movielens/movies")

In [0]:

MOVIES_PATH = dbutils.widgets.get("moviesPath") # passed from workflow or REST API
print (MOVIES_PATH)

abfss://bronze@gopaldatalake2.dfs.core.windows.net/movielens/movies


In [0]:
# how to create schema programatically instead of using inferSchema
from pyspark.sql.types import StructType, LongType, StringType, IntegerType, DoubleType

In [0]:
# True is nullable, False is non nullable
# give your own column name, datatypes
movieSchema = StructType()\
                .add("movieId", IntegerType(), True)\
                .add("title", StringType(), True)\
                .add("genres", StringType(), True)
     

In [0]:
# movieDf with schema we have, to avoid additional job creation, scan data overload
# since we provide schema, header = true, is to skip the first line in the csv
movieDf = (spark
            .read
            .option("header", True)
            .schema(movieSchema) # now , we don't run a job for schema
            .csv(MOVIES_PATH)
            )

# movieDf.printSchema()
# movieDf.show(5)
# make a note of number of jobs its create, compare with previous shell

In [0]:
# movie data, convert to parquet as is
# silver container
# movies directory

# move to silver container
# Put 
# Put MOVIE_TARGET_PATH as part of the widget

MOVIE_TARGET_PATH = dbutils.widgets.get("moviesTargetPath")
print("param path ", MOVIE_TARGET_PATH)

from datetime import datetime

now = datetime.now()

date_part = now.strftime("%d-%B-%Y").lower()   # 26-june-2026
time_part = now.strftime("%H-%M")              # 10:21

path = f"-{date_part}/{time_part}/"
MOVIE_TARGET_PATH = MOVIE_TARGET_PATH + "-" + path

print("dyamic  path ", MOVIE_TARGET_PATH)
# overwrite - delete old data
# write is a batch write
movieDf.write.mode("overwrite").parquet(MOVIE_TARGET_PATH)

param path  abfss://silver@gopaldatalake2.dfs.core.windows.net/movielens/movies
dyamic  path  abfss://silver@gopaldatalake2.dfs.core.windows.net/movielens/movies--26-june-2026/10-18/


In [0]:
# useful to pass result of one notebook to another notebook in jobs compute
# useful for dynamic path or pass one config to another notebook
# abfss://silver@gopaldatalake2.dfs.core.windows.net/movielens/movies/26-june-2026/10:21/

dbutils.jobs.taskValues.set(key = "movies_silver_path", value = MOVIE_TARGET_PATH)